# Jacket discounts: Cochran-Armitage trend test

**Cochran-Armitage Test for Trend** · a worked example from the StatNex teaching guide

**The story.** An online shop showed four groups of 40 visitors the same jacket at different discounts (none, 10%, 20%, 30%) and counted how many bought it.

**The question.** Does the purchase rate rise steadily as the discount grows?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/cochran_armitage.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Discount,Bought,Did not buy,Total
No discount,6.0,34.0,40.0
10% off,10.0,30.0,40.0
20% off,15.0,25.0,40.0
30% off,21.0,19.0,40.0
Total,52.0,108.0,160.0

These cells save `totals.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile totals.py
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# A k x 2 table: one row per level, in the order the levels rise (the order is the
# hypothesis), the first column the responders and the second the non-responders.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Bought", "Did not buy"]
table = data.set_index("Discount")[columns].apply(pd.to_numeric, errors="coerce")
# A row or column named like a total (Total, Sum, or a Japanese word for a total, as listed in totals.py) is a margin, not data,
# so it is left out, as the app does. Spaces, underscores, case and full-width letters
# are ignored when names are compared.
from totals import is_total

table = table.loc[[not is_total(row) for row in table.index],
                  [not is_total(column) for column in table.columns]]
# A row with a blank count is left out.
table = table.dropna()
counts = table.to_numpy(float)

## Observed frequency

In [ ]:
print(table)

## Cochran-Armitage test for trend

In [ ]:
# The levels are scored 1, 2, ..., k in row order -- not the doses written in the labels.
# Neither SciPy nor statsmodels has this test under its name, so it is written out:
#   z = sum s_i (r_i - n_i p) / sqrt(p (1 - p) [sum s_i^2 n_i - (sum s_i n_i)^2 / N])
# statsmodels' Table.test_ordinal_association is the same test with N - 1 in place of N
# (the Mantel-Haenszel form), so its z is smaller by sqrt((N - 1) / N).
# How to read it: z counts how many standard errors the trend in the response rate
# is from none; positive means the rate rises down the table. The test is built for a
# steady rise or fall, so a rate that goes up and then down can give z near 0 even when
# the levels clearly differ. It assumes independent cases and enough of them per level.
sizes = counts.sum(axis=1)
responses = counts[:, 0]
total = counts.sum()
overall = responses.sum() / total
scores = np.arange(1, len(counts) + 1)
numerator = (scores * (responses - sizes * overall)).sum()
variance = overall * (1 - overall) * ((scores**2 * sizes).sum() - (scores * sizes).sum() ** 2 / total)
z = numerator / np.sqrt(variance)
p_two = 2 * stats.norm.sf(abs(z))
# The one-sided P is half the two-sided one: the test in the direction the data point.
p_one = p_two / 2
trend = pd.Series({"Statistic: z": z, "Two-sided P": p_two, "One-sided P": p_one})
print()
print(trend)

## Effect size

In [ ]:
# Cramer's V from the ordinary chi-square of the whole table, without Yates' correction.
chi_square = stats.chi2_contingency(counts, correction=False).statistic
cramers_v = np.sqrt(chi_square / (total * (min(counts.shape) - 1)))
print(f"\nCramer's V = {cramers_v:.4f}")

## What the result says

Each group has 40 visitors. The number who bought rises at every step: 6 with no discount, 10 at 10% off, 15 at 20% off and 21 at 30% off.

The Cochran-Armitage test asks whether a rate moves steadily in one direction across ordered groups. Its statistic is z = 3.77. The two-sided P value is 0.00016. A P value is the chance of a trend at least this strong if the discount made no difference, so chance is a very poor explanation here. Cramer's V, a measure of the strength of the link from 0 to 1, is 0.30.

So yes, the purchase rate rises steadily with the discount. But more sales at a deeper discount do not automatically mean more profit.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.